In [1]:
import pandas as pd
import numpy as np

/Users/hganjoo/miniconda3/envs/sle/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
bat = pd.read_csv('test_batting_innings.csv')
bo = pd.read_csv('test_bowling_innings.csv')

/var/folders/hg/4pj6768x3q39vm215pfl2bcm0000gn/T/ipykernel_95170/2051112995.py:1: DtypeWarning: Columns (44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  bat = pd.read_csv('test_batting_innings.csv')
/var/folders/hg/4pj6768x3q39vm215pfl2bcm0000gn/T/ipykernel_95170/2051112995.py:2: DtypeWarning: Columns (42,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  bo = pd.read_csv('test_bowling_innings.csv')


In [36]:
matchballs = bo.groupby(['p_match','team_bat','inns']).team_innings_balls.max().reset_index().groupby(['team_bat','p_match'][::-1]).team_innings_balls.sum()
matchballs = bo.groupby(['p_match','team_bat']).agg({'balls':'sum','team_bowl':'max'}).reset_index()

In [37]:
matchballs

,p_match,team_bat,balls,team_bowl
0,62387,Australia,735,South Africa
1,62387,South Africa,866,Australia
2,62388,England,714,South Africa
3,62388,South Africa,649,England
4,62389,Australia,764,England
...,...,...,...,...
5207,1505259,Zimbabwe,618,Afghanistan
5208,1506012,Bangladesh,846,Ireland
5209,1506012,Ireland,976,Bangladesh
5210,1506013,Bangladesh,1261,Ireland


In [48]:
bat_balls = bat[bat.balls.notna()].groupby(['p_match','bat']).agg({'balls':'sum','team_bat':'max'}).reset_index()

In [49]:
bat_balls = bat_balls.merge(matchballs,on=['p_match','team_bat'],suffixes=('','_m'))

In [51]:
bat_balls['percbat'] = 100*(bat_balls.balls / bat_balls.balls_m)

In [77]:
batters = bat_balls.groupby('bat').agg({'balls_m':'count','percbat':'mean','balls':'sum'}).query('balls_m>49').reset_index()

In [78]:
bo_balls = bo.groupby(['p_match','bowl']).agg({'balls':'sum','team_bowl':'max'}).reset_index()
bo_balls = bo_balls.merge(matchballs,on=['p_match','team_bowl'],suffixes=('','_m'))

In [79]:
bo_balls['percbo'] = 100*(bo_balls.balls / bo_balls.balls_m)

In [80]:
bowlers = bo_balls.groupby('bowl').agg({'balls_m':'count','percbo':'mean','balls':'sum'}).query('balls_m>49').reset_index()

In [81]:
batters

,bat,balls_m,percbat,balls
0,A Flintoff,76,8.155096,6197.0
1,A Flower,63,15.533353,10636.0
2,A Kumble,119,5.359339,6455.0
3,A Ranatunga,81,11.541121,8957.0
4,AA Donald,66,2.823955,1856.0
...,...,...,...,...
244,Younis Khan,116,15.363359,19375.0
245,Yousuf Youhana,58,14.510652,8722.0
246,Z Crawley,64,11.408927,5475.0
247,Z Khan,84,2.917048,2381.0


In [82]:
mgd = batters.merge(bowlers,left_on='bat',right_on='bowl',suffixes=('_ba','_bo'))

In [85]:
mgd.shape,batters.shape,bowlers.shape

((100, 8), (249, 4), (134, 4))

In [87]:
mgd['totpart'] = mgd.percbat + mgd.percbo

In [89]:
mgd.sort_values('totpart')

,bat,balls_m_ba,percbat,balls_ba,bowl,balls_m_bo,percbo,balls_bo,totpart
93,V Sehwag,104,10.232718,10441.0,V Sehwag,69,4.895984,3731,15.128703
62,Mohammad Ashraful,61,11.810784,5940.0,Mohammad Ashraful,52,3.909499,1733,15.720283
78,SC Ganguly,111,12.034993,14070.0,SC Ganguly,75,3.864412,3117,15.899405
91,TM Dilshan,84,10.199426,8379.0,TM Dilshan,56,6.022139,3385,16.221566
57,ME Waugh,128,11.782695,15360.0,ME Waugh,94,4.755163,4853,16.537858
...,...,...,...,...,...,...,...,...,...
1,A Kumble,119,5.359339,6455.0,A Kumble,128,28.781820,40850,34.141159
61,Mehidy Hasan Miraz,56,8.997290,4299.0,Mehidy Hasan Miraz,55,25.326289,13164,34.323579
88,Shakib Al Hasan,70,11.807207,7429.0,Shakib Al Hasan,69,22.749114,15291,34.556320
53,M Muralidaran,101,1.775269,1721.0,M Muralidaran,121,33.091104,40863,34.866373


In [91]:
mgd.percbo.mean()

18.377637201209033